In [1]:
!pip install -q sentence-transformers datasets transformers huggingface-hub accelerate

from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login
import torch, os, json, random


In [2]:
HF_TOKEN = "your_token"
login(HF_TOKEN)
HF_USERNAME = "ij98"  # e.g. "imama"


In [3]:
def make_pair(resume, jd, score):
    return {"resume": resume, "job": jd, "score": float(score)}

# Example resumes & job descriptions (replace with your real dataset)
resumes = [
    "Experienced Python developer with 3 years in data analysis, pandas and scikit-learn, built ML pipelines.",
    "NLP engineer experienced in transformers, Hugging Face, and fine-tuning language models.",
    "Software engineer skilled in Java, Spring, and backend services; limited ML experience."
]

jobs = [
    "We need a Data Scientist skilled in Python, pandas, scikit-learn to build ML pipelines and analyze data.",
    "Seeking an NLP Engineer with experience fine-tuning transformers and using Hugging Face.",
    "Backend developer role requiring Java and Spring framework experience."
]

pairs = [
    make_pair(resumes[0], jobs[0], 1.0),  # good match
    make_pair(resumes[1], jobs[1], 1.0),
    make_pair(resumes[2], jobs[2], 1.0),
    make_pair(resumes[0], jobs[1], 0.2),  # not great match
    make_pair(resumes[1], jobs[0], 0.3),
    make_pair(resumes[2], jobs[0], 0.1)
]

dataset = Dataset.from_list(pairs).train_test_split(test_size=0.33, seed=42)
dataset


DatasetDict({
    train: Dataset({
        features: ['resume', 'job', 'score'],
        num_rows: 4
    })
    test: Dataset({
        features: ['resume', 'job', 'score'],
        num_rows: 2
    })
})

In [5]:
def to_input_examples(ds):
    examples = []
    for row in ds:
        # We will treat score as relevance; for contrastive training we create pairs: (resume, job) with label
        examples.append(InputExample(texts=[row['resume'], row['job']], label=row['score']))
    return examples

train_examples = to_input_examples(dataset['train'])
test_examples = to_input_examples(dataset['test'])
len(train_examples), len(test_examples)


(4, 2)

In [6]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"  # small & effective
model = SentenceTransformer(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
batch_size = 16

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
# If you have soft labels use CosineSimilarityLoss
loss_fct = losses.CosineSimilarityLoss(model)


In [8]:
# fine tuning encoder:
num_epochs = 3
warmup_steps = max(100, int(len(train_dataloader) * num_epochs * 0.1))

model.fit(
    train_objectives=[(train_dataloader, loss_fct)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path="./output/resume-job-encoder"
)


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ijawad-bese16seecs (ml_saad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [9]:
# Load test dataset and compute similarities
encoder = SentenceTransformer("./output/resume-job-encoder")

def evaluate_pairs(examples):
    resumes = [e.texts[0] for e in examples]
    jobs = [e.texts[1] for e in examples]
    scores_true = [e.label for e in examples]
    emb_res = encoder.encode(resumes, convert_to_tensor=True)
    emb_jobs = encoder.encode(jobs, convert_to_tensor=True)
    cos_scores = util.cos_sim(emb_res, emb_jobs).diagonal().cpu().tolist()  # pairwise diagonal
    # Basic metrics
    import numpy as np
    mse = float(((np.array(cos_scores)-np.array(scores_true))**2).mean())
    return {"mse": mse, "avg_cos": float(sum(cos_scores)/len(cos_scores)) , "cos_scores": cos_scores, "true": scores_true}

eval_results = evaluate_pairs(test_examples)
eval_results


{'mse': 0.0634837330921174,
 'avg_cos': 0.47286535799503326,
 'cos_scores': [0.29039815068244934, 0.6553325653076172],
 'true': [0.2, 1.0]}

In [10]:
repo_id = f"{HF_USERNAME}/resume-job-encoder2"
encoder.save_pretrained("./hf_model")

encoder.push_to_hub(repo_id, exist_ok=True)
print("Pushed to HF Hub:", repo_id)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tl2nohj/model.safetensors:   2%|2         | 2.15MB / 90.9MB            

Pushed to HF Hub: ij98/resume-job-encoder2


In [11]:
# local inference
encoder = SentenceTransformer("./output/resume-job-encoder")
def similarity_score(resume_text, job_text):
    emb_r = encoder.encode(resume_text, convert_to_tensor=True)
    emb_j = encoder.encode(job_text, convert_to_tensor=True)
    return float(util.cos_sim(emb_r, emb_j).item())

print(similarity_score(resumes[0], jobs[0]))


0.8585694432258606


In [12]:
!pip install -q PyPDF2

import PyPDF2

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file."""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text.strip()

def match_pdf_files(resume_pdf, job_pdf):
    """
    Match resume PDF to job description PDF.

    Args:
        resume_pdf: Path to resume PDF
        job_pdf: Path to job description PDF

    Returns:
        Similarity score (0-1)
    """
    # Extract text
    resume_text = extract_text_from_pdf(resume_pdf)
    job_text = extract_text_from_pdf(job_pdf)

    # Use existing model
    emb_r = encoder.encode(resume_text, convert_to_tensor=True)
    emb_j = encoder.encode(job_text, convert_to_tensor=True)

    score = float(util.cos_sim(emb_r, emb_j).item())

    print(f"📄 Resume Preview:\n{resume_text[:200]}...\n")
    print(f"💼 Job Preview:\n{job_text[:200]}...\n")
    print(f"🎯 Similarity Score: {score:.3f}")

    return score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.5 MB/s eta 0:00:00


In [13]:

score = match_pdf_files('CV_Example (1).pdf', 'NLP Engineer_Post.pdf')

📄 Resume Preview:
Isabella
 
Kim
 
isabella@kim.com
 
•
 
(557)
 
300-9285
 
•
 
linkedin.com/in/isabella-kim
 
•
 
@isabella.kim
 
NLP
 
Engineer
 
Seasoned
 
NLP
 
Engineer
 
with
 
9
 
years
 
developing
 
machine
 ...

💼 Job Preview:
NLP
 
Engineer
 
Pozent
 
Corpor ation
 
 
South
 
Plainﬁeld,
 
NJ
 
Role
 
Overview
 
 
Develop
 
and
 
deplo y
 
NLP
 
solutions
 
using
 
transformers,
 
text
 
mining
 
techniques,
 
and
 
modern
...

🎯 Similarity Score: 0.676


In [14]:
score = match_pdf_files('CV_Example (1).pdf', 'GraphicDesign_Post.pdf')

📄 Resume Preview:
Isabella
 
Kim
 
isabella@kim.com
 
•
 
(557)
 
300-9285
 
•
 
linkedin.com/in/isabella-kim
 
•
 
@isabella.kim
 
NLP
 
Engineer
 
Seasoned
 
NLP
 
Engineer
 
with
 
9
 
years
 
developing
 
machine
 ...

💼 Job Preview:
Graphic
 
Designer
 
Paula's
 
Choice
 
Skincar e
 
 
United
 
States
 
Wher e
 
Trust
 
Leads,
 
Bold
 
Ideas
 
Grow,
 
and
 
Community
 
Thriv es
 
 
Paula’s
 
Choice,
 
a
 
Global
 
Skincar e
 
Lea...

🎯 Similarity Score: 0.278


In [16]:
#The NLP Candidate matches with a score of 0.278 with a graphic design post vs a score of 0.676 for a NLP Post
